In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

## 1. Configuration

In [ ]:
# Paths
DATA_DIR = Path("./data")
MODEL_DIR = Path("../backend/app/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Heat ranges by variety (SHU)
VARIETY_HEAT_RANGES = {
    "siling_haba": (500, 15000),
    "siling_labuyo": (15000, 100000),
    "super_labuyo": (50000, 150000)
}

# Heat categories
HEAT_CATEGORIES = {
    "Mild": (0, 5000),
    "Medium": (5001, 15000),
    "Hot": (15001, 50000),
    "Extra Hot": (50001, 500000)
}

## 2. Generate Synthetic Training Data

In [ ]:
def generate_synthetic_data(n_samples=1000):
    """Generate synthetic training data based on known chili characteristics."""
    np.random.seed(42)
    data = []
    
    for variety, (min_shu, max_shu) in VARIETY_HEAT_RANGES.items():
        n_variety = n_samples // len(VARIETY_HEAT_RANGES)
        
        for _ in range(n_variety):
            # Generate SHU with normal distribution within range
            shu = np.clip(
                np.random.normal((min_shu + max_shu) / 2, (max_shu - min_shu) / 4),
                min_shu, max_shu
            )
            
            # Generate correlated features
            # Higher heat correlates with: smaller size, darker color, higher capsaicin markers
            heat_factor = (shu - min_shu) / (max_shu - min_shu)
            
            # Size features (inverse correlation with heat)
            if variety == "siling_haba":
                length = np.random.normal(80 - heat_factor * 20, 10)
                width = np.random.normal(15 - heat_factor * 3, 2)
            elif variety == "siling_labuyo":
                length = np.random.normal(30 - heat_factor * 10, 5)
                width = np.random.normal(8 - heat_factor * 2, 1)
            else:  # super_labuyo
                length = np.random.normal(25 - heat_factor * 8, 4)
                width = np.random.normal(6 - heat_factor * 1.5, 0.8)
            
            # Color features
            red_intensity = 150 + heat_factor * 80 + np.random.normal(0, 10)
            green_intensity = 100 - heat_factor * 50 + np.random.normal(0, 10)
            hue = 10 + heat_factor * 15 + np.random.normal(0, 3)
            saturation = 0.6 + heat_factor * 0.3 + np.random.normal(0, 0.05)
            
            # Texture features
            texture_roughness = 0.3 + heat_factor * 0.4 + np.random.normal(0, 0.05)
            surface_glossiness = 0.7 - heat_factor * 0.2 + np.random.normal(0, 0.05)
            
            # Maturity (affects heat)
            maturity_score = np.random.uniform(0.4, 1.0)
            
            # Variety encoding
            variety_encoded = list(VARIETY_HEAT_RANGES.keys()).index(variety)
            
            data.append({
                'variety': variety,
                'variety_encoded': variety_encoded,
                'length_mm': max(10, length),
                'width_mm': max(3, width),
                'aspect_ratio': max(10, length) / max(3, width),
                'red_intensity': np.clip(red_intensity, 0, 255),
                'green_intensity': np.clip(green_intensity, 0, 255),
                'hue': np.clip(hue, 0, 180),
                'saturation': np.clip(saturation, 0, 1),
                'texture_roughness': np.clip(texture_roughness, 0, 1),
                'surface_glossiness': np.clip(surface_glossiness, 0, 1),
                'maturity_score': maturity_score,
                'area_mm2': max(10, length) * max(3, width) * 0.8,
                'shu': shu
            })
    
    return pd.DataFrame(data)

df = generate_synthetic_data(3000)
print(f"Generated {len(df)} samples")
df.head()

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# SHU distribution by variety
for variety in VARIETY_HEAT_RANGES.keys():
    subset = df[df['variety'] == variety]
    axes[0, 0].hist(subset['shu'], bins=30, alpha=0.5, label=variety)
axes[0, 0].set_xlabel('Scoville Heat Units')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('SHU Distribution by Variety')
axes[0, 0].legend()

# Size vs Heat
scatter = axes[0, 1].scatter(df['length_mm'], df['shu'], c=df['variety_encoded'], 
                              cmap='viridis', alpha=0.5)
axes[0, 1].set_xlabel('Length (mm)')
axes[0, 1].set_ylabel('SHU')
axes[0, 1].set_title('Size vs Heat Level')

# Color vs Heat
axes[1, 0].scatter(df['red_intensity'], df['shu'], c=df['variety_encoded'], 
                   cmap='viridis', alpha=0.5)
axes[1, 0].set_xlabel('Red Intensity')
axes[1, 0].set_ylabel('SHU')
axes[1, 0].set_title('Color vs Heat Level')

# Correlation heatmap
feature_cols = ['length_mm', 'width_mm', 'red_intensity', 'saturation', 
                'texture_roughness', 'maturity_score', 'shu']
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Feature Correlations')

plt.tight_layout()
plt.savefig('./heat_data_analysis.png', dpi=150)
plt.show()

## 3. Prepare Features

In [ ]:
# Define features
FEATURE_COLS = [
    'variety_encoded',
    'length_mm',
    'width_mm',
    'aspect_ratio',
    'red_intensity',
    'green_intensity',
    'hue',
    'saturation',
    'texture_roughness',
    'surface_glossiness',
    'maturity_score',
    'area_mm2'
]

X = df[FEATURE_COLS].values
y = df['shu'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 4. Train Models

In [ ]:
# Define models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, 
        max_depth=15, 
        min_samples_split=5,
        random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    ),
    'Decision Tree': DecisionTreeRegressor(
        max_depth=10,
        min_samples_split=5,
        random_state=42
    )
}

results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for linear models
    if 'Linear' in name or 'Ridge' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    })
    
    print(f"  RMSE: {rmse:.2f} SHU")
    print(f"  MAE: {mae:.2f} SHU")
    print(f"  R²: {r2:.4f}")

# Results summary
df_results = pd.DataFrame(results)
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(df_results.to_string(index=False))

In [ ]:
# Visualize results
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(df_results))
width = 0.35

bars1 = ax.bar(x - width/2, df_results['R²'], width, label='R² Score', color='steelblue')

ax.set_ylabel('R² Score')
ax.set_xlabel('Model')
ax.set_title('Model Comparison - R² Score')
ax.set_xticks(x)
ax.set_xticklabels(df_results['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('./model_comparison.png', dpi=150)
plt.show()

## 5. Feature Importance

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
importances = rf_model.feature_importances_

feature_importance = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for SHU Prediction')
plt.tight_layout()
plt.savefig('./feature_importance.png', dpi=150)
plt.show()

## 6. Hyperparameter Tuning

In [ ]:
# Grid search for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Performing GridSearch for Random Forest...")
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best R² score: {grid_search.best_score_:.4f}")

In [ ]:
# Train final model with best parameters
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)

print("\nFinal Random Forest Performance:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.2f} SHU")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_best):.2f} SHU")
print(f"  R²: {r2_score(y_test, y_pred_best):.4f}")

## 7. Save Models

In [ ]:
# Save best Random Forest model
joblib.dump(best_rf, MODEL_DIR / 'heat_predictor_rf.pkl')
print(f"Random Forest model saved to {MODEL_DIR / 'heat_predictor_rf.pkl'}")

# Save Linear Regression model
lr_model = models['Linear Regression']
joblib.dump(lr_model, MODEL_DIR / 'heat_predictor_lr.pkl')
print(f"Linear Regression model saved to {MODEL_DIR / 'heat_predictor_lr.pkl'}")

# Save scaler
joblib.dump(scaler, MODEL_DIR / 'heat_scaler.pkl')
print(f"Scaler saved to {MODEL_DIR / 'heat_scaler.pkl'}")

# Save feature names
import json
with open(MODEL_DIR / 'heat_features.json', 'w') as f:
    json.dump({
        'feature_columns': FEATURE_COLS,
        'heat_categories': HEAT_CATEGORIES,
        'variety_heat_ranges': VARIETY_HEAT_RANGES
    }, f, indent=2)

print("\nAll models saved successfully!")

## 8. Prediction Demo

In [ ]:
def predict_heat(features, model, use_scaled=False, scaler=None):
    """Predict heat level from features."""
    if use_scaled and scaler is not None:
        features = scaler.transform([features])
    else:
        features = [features]
    
    shu = model.predict(features)[0]
    
    # Determine category
    for category, (min_val, max_val) in HEAT_CATEGORIES.items():
        if min_val <= shu <= max_val:
            return shu, category
    
    return shu, "Unknown"

# Example prediction
sample_features = [1, 28, 7, 4.0, 200, 60, 18, 0.8, 0.6, 0.5, 0.85, 180]
shu, category = predict_heat(sample_features, best_rf)

print(f"\nPrediction Demo:")
print(f"  Predicted SHU: {shu:.0f}")
print(f"  Heat Category: {category}")